# Data Loading

In [1]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("../data/olj-jobs.db")

query = "select * from jobs"

df = pd.read_sql_query(query, conn)

conn.close()

In [2]:
df.head()

,id,job_id,title,work_type,salary,hours_per_week,job_overview,summary,link,raw_text,date_created
0,1,1462058,Back-End Developer - OnlineJobs.ph,Full Time,600-1000,40,Responsibilities:\n\n\n\nDevelop and maintain ...,💻 **Back-End Developer**\n\n**Focus:** Server-...,https://www.onlinejobs.ph/jobseekers/job/1462058,"<!DOCTYPE html>\n<html lang=""en"">\n <head>\...",2025-09-06T13:56:44.093821
1,2,1462056,Front-End Developer - OnlineJobs.ph,Full Time,600,40,Responsibilities:\n\nDevelop and implement fro...,💻 Front-End React Developer (Web Application)\...,https://www.onlinejobs.ph/jobseekers/job/1462056,"<!DOCTYPE html>\n<html lang=""en"">\n <head>\...",2025-09-06T13:56:49.196364
2,3,1463592,Youtube Video Editor (Long-Form) - OnlineJobs.ph,Any,N/A,TBD,About Us:\n\r\nWe run a YouTube channel focuse...,🎬 YouTube Video Editor (Long-Form Content)\n\n...,https://www.onlinejobs.ph/jobseekers/job/1463592,"<!DOCTYPE html>\n<html lang=""en"">\n <head>\...",2025-09-06T13:56:53.761090
3,4,1455562,ECommerce Video Editor - OnlineJobs.ph,Any,$160 minimum ++,5,Video Editor – $10 Per Video – Korean Skincare...,🎥 Video Editor (Korean Skincare Brand)\n\n**Ty...,https://www.onlinejobs.ph/jobseekers/job/1455562,"<!DOCTYPE html>\n<html lang=""en"">\n <head>\...",2025-09-06T23:15:15.608966
4,5,1463739,Sales Call Closer (Perfect American Accent Req...,Full Time,800,40,We are a fast-growing marketing agency helping...,🚀 **Sales Closer (Inbound Calls)** 🚀\n\n**Focu...,https://www.onlinejobs.ph/jobseekers/job/1463739,"<!DOCTYPE html>\n<html lang=""en"">\n <head>\...",2025-09-06T23:15:22.270302


In [ ]:
# remove this when doing the actual cleaning, the cosine similarity takes 20gb + ram so we use a sample for testing purposes 
df = df.sample(frac=0.05)
len(df)

2929

# Dropping rows where salary cannot be determined if USD or PHP
while we can wrangle the data here to retain more rows I don't want to spend a lot of time doing that. hence, only salaries where we can explicitly determine if its USD or PHP are retained.



In [4]:
import re


def is_usd_or_php(salary):
    if pd.isna(salary):
        return False

    s = str(salary).strip()

    non_usd_dollar_patterns = [
        r"\bAUD\b", r"\baud\b", r"A\$", r"AU\$",
        r"\bCAD\b", r"\bcad\b", r"C\$", r"CA\$",
        r"\bSGD\b", r"\bsgd\b", r"S\$",
        r"\bHKD\b", r"\bhkd\b", r"HK\$",
        r"\bNZD\b", r"\bnzd\b", r"NZ\$",
        r"\bMXN\b", r"\bmxn\b", r"MX\$",
        r"\bBRL\b", r"\bbrl\b", r"R\$",
    ]

    usd_patterns = [
        r"\bUSD\b",
        r"\bUsd\b",
        r"\busd\b",
        r"US\$",
        r"U\.S\.\$",
    ]

    php_patterns = [
        r"\bPHP\b",
        r"\bphp\b",
        r"\bPhp\b",
        r"\bPeso\b",
        r"\bpeso\b",
        r"Philippine",
        r"₱",
        r"\?",
    ]

    for p in non_usd_dollar_patterns:
        if re.search(p, s):
            return False

    for p in usd_patterns:
        if re.search(p, s):
            return True

    if re.search(r"\$", s):
        return True

    for p in php_patterns:
        if re.search(p, s):
            return True

    return False


mask_valid_salary = df["salary"].apply(is_usd_or_php)
count_dropped = (~mask_valid_salary).sum()
print(f"Dropping {count_dropped} rows where salary currency cannot be determined.")
df_cleaned_salary = df[mask_valid_salary].copy()

Dropping 1362 rows where salary currency cannot be determined.


In [5]:
df_cleaned_salary["salary"].sample(15)

4448                                  $960/ Month
46105                                 $1100/Month
17435    $2.00 (per hour for probationary period)
5442                                   PHP 20,000
2289                              $420-$750/month
55036                           $600-$800 / Month
484                                     $200-1000
4730                         Starting at 1600 USD
39876                                  $100/month
42610                                $15 per reel
20467      700$-1000$/2 weeks/pay 2 times/1 month
51057                                        $400
34114                            Starts at $10/hr
7190                                         500$
44430                                  52,000 PHP
Name: salary, dtype: str

# Duplicate Removal

In [6]:
duplicate_rows = df_cleaned_salary[df_cleaned_salary.duplicated("job_id")]
count = df_cleaned_salary.duplicated("job_id").sum()
print(f"Found {count} duplicates.")

Found 0 duplicates.


In [7]:
duplicate_rows = df_cleaned_salary[df_cleaned_salary.duplicated("title")]
count = df_cleaned_salary.duplicated("title").sum()
print(f"Found {count} duplicates.")

Found 0 duplicates.


In [8]:
duplicate_rows = df_cleaned_salary[df_cleaned_salary.duplicated("raw_text")]
count = df_cleaned_salary.duplicated("raw_text").sum()
print(f"Found {count} duplicates.")

Found 0 duplicates.


removed using cosine similarity since some listings are duplicated but with same job titles and a slightly tweaked job overview when I was looking through the data, it had the same skills, salary, poster, etc, but with like a word or two added to it

In [9]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


def duplicated_fuzzy(df, column, threshold=0.80):
    texts = df[column].fillna("")
    vectorizer = TfidfVectorizer(stop_words="english")
    tfidf_matrix = vectorizer.fit_transform(texts)

    cosine_sim = cosine_similarity(tfidf_matrix)
    upper_tri = np.triu(cosine_sim, k=1)

    duplicate_indices = [
        i for i in range(upper_tri.shape[1]) if any(upper_tri[:, i] > threshold)
    ]

    mask = np.zeros(len(df), dtype=bool)
    mask[duplicate_indices] = True

    return pd.Series(mask, index=df.index)

In [10]:
is_duplicate = duplicated_fuzzy(df_cleaned_salary, "job_overview", threshold=0.80)
count = is_duplicate.sum()
print(f"Found {count} near-duplicates.")

Found 29 near-duplicates.


In [11]:
df_cleaned_job_overview = df_cleaned_salary[~is_duplicate]
print(
    f"Original rows: {len(df_cleaned_salary)} | Cleaned rows: {len(df_cleaned_job_overview)}"
)

Original rows: 1567 | Cleaned rows: 1538


# Exporting

In [ ]:
import sqlite3

new_conn = sqlite3.connect("./olj-jobs-cleaned.db")

df_cleaned_job_overview.to_sql("olj_jobs", new_conn, if_exists="replace", index=False)

new_conn.close()